# Setup Ollama - Colab

In [80]:
# %pip install colab-xterm
# %load_ext colabxterm

Lanch xtrem terminal in window.

%xterm

Download and run ollama server

curl -sSL https://ollama.ai/install.sh | sh && ollama serve

In [81]:
# !curl http://localhost:11434/api/pull -d '{  "model": "gemma3:12b" }'

In [82]:
# !curl http://localhost:11434/api/pull -d '{  "model": "nomic-embed-text" }'

# Setup tools

In [83]:
# %pip install -qU pandas langchain-ollama langchain-community langchain-text-splitters pypdf

In [84]:
import pandas as pd

from langchain.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain.prompts import PromptTemplate
from langchain.base_language import BaseLanguageModel
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from pathlib import Path

In [ ]:
class CFG:
    model = "gemma3:12b"
    model_embed = "nomic-embed-text"
    rag_file = Path("ustawa_o_sygnalistach.pdf")
    questions_file = "train.csv"
    llm_models = [
        "qwen2.5:14b",
        "gemma3:12b",
        "SpeakLeash/bielik-11b-v2.3-instruct:Q4_K_M",
        "deepseek-r1:14b",
    ]


AI_PROMPT = """" 
Jesteś modelem AI, który odpowiada na pytania użytkownika wyłącznie w formie liczbowej.

Użytkownik poda dwa elementy:

Question: pytanie, na które należy odpowiedzieć.
Context: kontekst zawierający dane potrzebne do odpowiedzi.

---

Zasady:

Jeśli w Context znajduje się wartość liczbowa, zwracasz ją jako odpowiedź.
Jeśli Context zawiera przedział liczbowy (np. 10–20), zwracasz maksymalną wartość, chyba że Question wprost wymaga minimum (np. „Jaka jest najmniejsza dopuszczalna kara?” – wtedy zwracasz wartość minimalną).
Jeśli Context zawiera różne liczby, wybierasz tę najbardziej adekwatną do Question.
Jeśli liczba nie jest podana wprost, ale można ją oszacować na podstawie treści (np. „kilka tysięcy” = 2000), zwracasz wartość szacunkową.
Jeśli w Context nie ma liczby, zwracasz „Brak danych”.
Nie podajesz żadnych wyjaśnień, tylko liczbę.

---

Przykłady:

Przykład 1:
Question: Ile osób mieszka w tym mieście?
Context: Populacja miasta wynosi 50 000.
Answer: 50000

Przykład 2:
Question: Jaka jest temperatura wrzenia tej cieczy?
Context: Temperatura wrzenia wynosi 80–100°C.
Answer: 100

Przykład 3:
Question: Jaka jest najmniejsza dopuszczalna kara?
Context: Kara wynosi od 500 do 5000 zł.
Answer: 500

Przykład 4:
Question: Ile lat miał najstarszy uczestnik?
Context: Wiek uczestników wynosił od 25 do 60 lat.
Answer: 60

Przykład 5:
Question: Ile stron ma ta książka?
Context: Brak informacji o liczbie stron.
Answer: Brak danych

---

Question: {question} 
Context: {context}
Answer:
"""

# Load models

In [86]:
try:
    llm = OllamaLLM(model=CFG.model)
    llm_embed = OllamaEmbeddings(model=CFG.model_embed)
except ModuleNotFoundError as e:
    print("Please install ollama first.")
    print(e.msg)
    exit()

# Data preparation

## Load questions

In [ ]:
df_qa = pd.read_csv(CFG.questions_file)
df_qa["question"] = df_qa["question"].str.strip().str.replace(r"\s+", " ", regex=True)

df_qa.head()

,id,question
0,q1,W ciągu ilu dni od przedstawienia projektu procedury zgłoszeń wewnętrznych przez podmiot prawny muszą zakończyć się konsultacje?
1,q2,Od ilu zatrudnionych osób podmiot prawny musi ustanowić wewnętrzną procedurę zgłoszeń naruszeń?
2,q3,W jakim terminie (dni) należy potwierdzić sygnaliście przyjęcie zgłoszenia wewnętrznego?
3,q4,Po ilu miesiącach od upływu terminu kalendarzowego Rzecznik Praw Obywatelskich usuwa dane osobowe związane ze zgłoszeniem zewnętrznym?
4,q5,Przez ile lat podmiot prawny i organ publiczny przechowują dane osobowe związane ze zgłoszeniem?


## Load PDF documents

Documents represent each page in file.

In [88]:
loader = PyPDFLoader(CFG.rag_file)
documents = loader.load()
documents[0]

Document(metadata={'producer': 'Microsoft® Word dla Microsoft 365', 'creator': 'Microsoft® Word dla Microsoft 365', 'creationdate': '2024-06-25T09:38:38+02:00', 'title': 'Ustawa z dnia 14 czerwca 2024 r. o ochronie sygnalistów', 'author': 'RCL', 'moddate': '2024-06-25T09:38:38+02:00', 'source': 'ustawa_o_sygnalistach.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='©Kancelaria Sejmu    s. 1/17 \n      \n \n2024-06-25 \n \n \nDz. U. 2024 poz. 928 \n \n \nUSTAWA  \nz dnia 14 czerwca 2024 r. \no ochronie sygnalistów1), 2) \nRozdział 1 \nPrzepisy ogólne \nArt. 1. Ustawa reguluje: \n1) warunki objęcia ochroną sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n2) środki ochrony sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n3) zasady ustalania wewnętrznej procedury zgłaszania informacji o naruszeniach prawa i podejmowania działań następczych; \n4) zasady zgłaszania informacji o naruszenia

### Split documents into smaller chunks

In [89]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=125)
all_splits = text_splitter.split_documents(documents)

all_splits[0]

Document(metadata={'producer': 'Microsoft® Word dla Microsoft 365', 'creator': 'Microsoft® Word dla Microsoft 365', 'creationdate': '2024-06-25T09:38:38+02:00', 'title': 'Ustawa z dnia 14 czerwca 2024 r. o ochronie sygnalistów', 'author': 'RCL', 'moddate': '2024-06-25T09:38:38+02:00', 'source': 'ustawa_o_sygnalistach.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='©Kancelaria Sejmu    s. 1/17 \n      \n \n2024-06-25 \n \n \nDz. U. 2024 poz. 928 \n \n \nUSTAWA  \nz dnia 14 czerwca 2024 r. \no ochronie sygnalistów1), 2) \nRozdział 1 \nPrzepisy ogólne \nArt. 1. Ustawa reguluje: \n1) warunki objęcia ochroną sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n2) środki ochrony sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n3) zasady ustalania wewnętrznej procedury zgłaszania informacji o naruszeniach prawa i podejmowania działań następczych; \n4) zasady zgłaszania informacji o naruszenia

# Vector Database

In [90]:
vector_db = InMemoryVectorStore(embedding=llm_embed)
vector_db.add_documents(all_splits)

['08e06f9c-ba23-4815-b990-940a8326d224',
 'e99fcef0-fe72-47eb-863b-f36f3a7c3fde',
 '34342119-8d2f-45a5-8620-b0c233c7251c',
 '07fb29b4-863c-41b9-99d7-2b680571cdc7',
 'db564a80-554c-496c-9d92-947080ff6774',
 '8b86b6d8-d69e-4eb4-b478-40643c16ef23',
 '8fe8e9a3-e77f-4357-b8f0-88edcc956b71',
 '63ccecec-da4d-4cce-b250-1bc8ebd6d4b4',
 'c6dfe8da-7bc6-4b4e-984a-aa271c44fdb8',
 '1c650368-435d-4ff6-a280-57d73c355c98',
 '63a85a3a-0f7f-4335-8f9b-50038b3bb86d',
 'a0f3edb7-c5b8-4ba4-9283-78b36152a76b',
 'c55e771e-8bf3-41ea-b17a-83f2bfaaa769',
 'd9ceb1b3-c221-4ff2-a4ad-88e233bf7e69',
 'eced0e27-5314-4394-9454-5a82dac7f169',
 '682b2e8b-c8c2-458d-9812-9f73fd4e2481',
 'f70b4b3c-3017-4d2e-988d-804192667750',
 'c3c1c717-0b11-4a58-a017-524cd0f46821',
 '70f1905f-97f9-4382-9074-8d87356edbee',
 '486a044a-841f-4f8c-8e88-d02ff84942ee',
 'd1ee966c-dfc7-4e54-bdb7-956f0e215fe4',
 '2f440b6b-d92b-4388-896d-ecf52fcad858',
 'ebfb86ca-384a-4542-ae95-d6378fa4b2b1',
 '7c049b17-569c-46d3-a45d-b1f3b5f47e46',
 '8183b0b8-1ca1-

# Prepare prompt

In [91]:
prompt = PromptTemplate(template=AI_PROMPT, input_variables=["context", "question"])

# QA

## Extra functions

In [92]:
def format_docs(docs):
    result = "\n\n".join(doc.page_content for doc in docs)
    return result

In [93]:
def prepare_chain(llm: BaseLanguageModel):
    return (
        {
            "context": vector_db.as_retriever(
                search_kwargs={"score_threshold": 0.9},
            )
            | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

In [94]:
# df_qa["answer"] = df_qa['question'].apply(lambda q: prepare_chain(llm).invoke(q))

In [95]:
# df_qa

In [ ]:
for model in CFG.llm_models:
    llm = OllamaLLM(model=model)
    df_qa[f"answer{model}"] = df_qa["question"].apply(
        lambda q: prepare_chain(llm).invoke(q)
    )

KeyboardInterrupt: 

In [ ]:
pd.set_option("display.max_colwidth", None)
df_qa

,id,question,answerqwen2.5:14b,answergemma3:12b,answerSpeakLeash/bielik-11b-v2.3-instruct:Q4_K_M
0,q1,W ciągu ilu dni od przedstawienia projektu procedury zgłoszeń wewnętrznych przez podmiot prawny muszą zakończyć się konsultacje?,10,10,"5. Podmiot prawny ustala procedurę zgłoszeń wewnętrznych po konsultacjach z: \r\n1) zakładową organizacją związkową albo zakładowymi organizacjami związkowymi, jeżeli w podmiocie prawnym działają takie organizacje; \r\n2) przedstawicielami pracowników, wyłonionymi w trybie przyjętym u danego podmiotu prawnego, jeżeli w podmiocie prawnym nie działa zakładowa organizacja związkowa albo zakładowe organizacje związkowe. \r\n\r\nW przypadku braku wyłonienia przedstawicieli pracowników, procedura zgłoszeń wewnętrznych jest ustalana przez podmiot prawny bez konsultacji z przedstawicielami pracowników."
1,q2,Od ilu zatrudnionych osób podmiot prawny musi ustanowić wewnętrzną procedurę zgłoszeń naruszeń?,50,50,"Odpowiedź na pytanie ""Ile osób musi być zatrudnionych w podmiocie prawnym, aby był on zobowiązany do ustanowienia procedury zgłoszeń wewnętrznych?"":\r\n\r\nZgodnie z treścią tekstu, podmiot prawny jest zobowiązany do ustanowienia procedury zgłoszeń wewnętrznych, jeśli zatrudnia co najmniej 50 osób. Wynika to z faktu, że wymóg ten dotyczy podmiotów prawnych, które zatrudniają odpowiednią liczbę pracowników (w tym przypadku co najmniej 50).\r\n\r\nWarto również zauważyć, że procedura zgłoszeń wewnętrznych powinna być ustalona po konsultacjach z odpowiednimi podmiotami, takimi jak zakładowa organizacja związkowa lub przedstawiciele osób świadczących pracę na rzecz podmiotu prawnego. Konsultacje te powinny trwać nie krócej niż 5 dni i nie dłużej niż 10 dni od dnia przedstawienia projektu procedury zgłoszeń wewnętrznych."
2,q3,W jakim terminie (dni) należy potwierdzić sygnaliście przyjęcie zgłoszenia wewnętrznego?,"Na podstawie przeczytanych przepisów prawnych, termin na przekazanie zgłoszenia zewnętrznego przez Rzecznika Praw Obywatelskich do właściwego organu publicznego wynosi **14 dni** od dnia dokonania zgłoszenia. Zatem odpowiedź to **14 dni**.",14,"W odpowiedzi na pytanie dotyczące przepisów ustawy o ochronie osób zgłaszających naruszenia prawa, chciałbym wyjaśnić kilka kluczowych kwestii: \r\n\r\n1. **Wstępna weryfikacja zgłoszenia zewnętrznego przez Rzecznika Praw Obywatelskich**: Zgodnie z art. 32 ustawy, Rzecznik Praw Obywatelskich przeprowadza wstępną weryfikację zgłoszenia zewnętrznego, aby ustalić, czy dotyczy ono informacji o naruszeniu prawa oraz zidentyfikować organ publiczny właściwy do podjęcia działań następczych. Jeśli zgłoszenie dotyczy naruszenia prawa, Rzecznik Praw Obywatelskich przekazuje je niezwłocznie, ale nie później niż w terminie 14 dni od dnia dokonania zgłoszenia, do odpowiedniego organu publicznego. Jeśli zgłoszenie nie dotyczy naruszenia prawa, Rzecznik Praw Obywatelskich informuje o tym sygnalistę i może wskazać inne tryby rozpatrzenia sprawy.\r\n\r\n2. **Przekazanie zgłoszenia zewnętrznego do organu publicznego**: Rzecznik Praw Obywatelskich przekazuje zgłoszenie zewnętrzne do właściwego organu publicznego, który jest odpowiedzialny za podjęcie działań następczych. Organ ten powinien niezwłocznie podjąć czynności mające na celu zbadanie sprawy i ustalenie, czy doszło do naruszenia prawa. Jeśli organ publiczny nie podejmie żadnych działań następczych ani nie przekaże sygnaliście informacji zwrotnej w określonym terminie, sygnalista może dokonać zgłoszenia zewnętrznego bezpośrednio do organu publicznego.\r\n\r\n3. **Odstąpienie od przekazania zgłoszenia**: Rzecznik Praw Obywatelskich może odstąpić od przekazania zgłoszenia zewnętrznego, jeśli ustali, że informacja objęta zgłoszeniem podlega rozpatrzeniu w trybie przewidzianym w innych przepisach, takich jak powództwo cywilne, zawiadomienie o podejrzeniu popełnienia przestępstwa, skarga do sądu administracyjnego, skarga, wniosek lub petycja. W takim przypadku Rzecznik Praw Obywatelskich informuje sygnalistę o możli